# Milestone 02 - Leakage Review and Split Design

## Project
Bank Account Fraud Detection - Deep Learning & MLOps

## Purpose

This notebook evaluates whether the dataset contains leakage risks, whether feature distributions change across time, and which train/validation/test split strategy best represents real deployment.

The final split must be chosen based on data integrity and deployment realism, not on whichever split later produces the best model score.

## Milestone 02 Questions

1. Could any feature contain information that would not be available at prediction time?
2. Are any features suspiciously close to the target or derived from future outcomes?
3. Does the distribution of important features change across `month`?
4. Does fraud prevalence change enough across time to justify chronological evaluation?
5. Should the project use chronological, stratified-random, or another split design?
6. How should the final test set be sealed to prevent model-selection leakage?

## Split Decision Rules

Before evaluating models, the split strategy will be selected using the following principles:

- Prefer a chronological split if `month` represents ordered time and meaningful temporal distribution shift is present.
- Use a random stratified split only if temporal ordering is not meaningful or cannot support a realistic deployment simulation.
- The final test set must represent unseen future or held-out data and must not be used for feature selection, hyperparameter tuning, threshold tuning, calibration, or model selection.
- Preprocessing must be fit only on the training data after the split is finalized.
- Validation data may be used for development decisions.
- Test data must remain sealed until the model, preprocessing, calibration, threshold policy, and evaluation procedure are frozen.
- No split will be selected simply because it produces better model performance.

## Milestone Boundary

This notebook may define and save split membership, but it will not perform model training or final preprocessing.

### 1. Add the environment and data-loading cell

In [2]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
RAW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "Base.csv"

print("Python:", sys.version.split()[0])
print("Project root:", PROJECT_ROOT)
print("Raw data path:", RAW_DATA_PATH)
print("File exists:", RAW_DATA_PATH.exists())

df = pd.read_csv(RAW_DATA_PATH)

print("\nDataset shape:", df.shape)
print("Target:", "fraud_bool")
print("Months:", sorted(df["month"].unique().tolist()))

Python: 3.11.9
Project root: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops
Raw data path: c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\data\raw\Base.csv
File exists: True

Dataset shape: (1000000, 32)
Target: fraud_bool
Months: [0, 1, 2, 3, 4, 5, 6, 7]


### 2. Create the initial prediction-time leakage review

In [3]:
leakage_review_rows = []

for col in df.columns:
    if col == "fraud_bool":
        role = "target"
        leakage_status = "exclude_from_features"
        review_reason = "Outcome label; never used as an input feature."

    elif col == "month":
        role = "temporal_split_field"
        leakage_status = "hold_for_split_review"
        review_reason = (
            "Represents ordered time. Review for split design before deciding "
            "whether it should also be a model feature."
        )

    elif col == "device_fraud_count":
        role = "candidate_predictor"
        leakage_status = "high_priority_review"
        review_reason = (
            "Feature name directly references fraud and could potentially "
            "encode outcome-related information. It is also constant in Base.csv."
        )

    elif col == "credit_risk_score":
        role = "candidate_predictor"
        leakage_status = "high_priority_review"
        review_reason = (
            "Derived risk score. Must verify that it is calculated using only "
            "information available before the fraud decision."
        )

    else:
        role = "candidate_predictor"
        leakage_status = "requires_definition_review"
        review_reason = (
            "Verify that the feature exists at prediction time and is not "
            "derived from future or post-outcome information."
        )

    leakage_review_rows.append({
        "column": col,
        "role": role,
        "leakage_status": leakage_status,
        "review_reason": review_reason,
    })

leakage_review = pd.DataFrame(leakage_review_rows)

display(leakage_review)

,column,role,leakage_status,review_reason
0,fraud_bool,target,exclude_from_features,Outcome label; never used as an input feature.
1,income,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
2,name_email_similarity,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
3,prev_address_months_count,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
4,current_address_months_count,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
5,customer_age,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
6,days_since_request,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
7,intended_balcon_amount,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
8,payment_type,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...
9,zip_count_4w,candidate_predictor,requires_definition_review,Verify that the feature exists at prediction t...


## Leakage Review Standard

The Bank Account Fraud Dataset Suite represents a bank-account-opening fraud detection setting and includes temporal dynamics.

For this project, a feature will not be considered safe merely because it appears in the dataset.

A candidate feature must satisfy all of the following conditions before being approved for modeling:

- It would plausibly be available at the moment the fraud-risk prediction is made.
- It must not depend on the final fraud label.
- It must not summarize information collected after the application outcome.
- It must not contain information from future time periods.
- Any derived score must be understood well enough to determine whether it was produced before or after the prediction decision.
- Temporal fields must be evaluated separately for split design and model-feature use.

### High-Priority Features

`device_fraud_count` requires special scrutiny because its name explicitly references known fraud outcomes. In `Base.csv` it is constant at zero, so it currently provides no predictive information regardless of its semantics.

`credit_risk_score` requires special scrutiny because it is a derived risk score. It should only be retained if its computation uses information available before the fraud decision.

`month` will first be treated as a temporal ordering variable for split design. Whether it is later included as a model feature will be decided separately.

All other predictors remain provisional until their meaning and prediction-time availability have been reviewed.

### Important Principle

Statistical association with `fraud_bool` is not proof of leakage. Conversely, a feature can leak information even when its simple correlation with the target is weak. Leakage must therefore be assessed using both feature semantics and empirical evidence.


### 3. Screen numerical features for unusually strong target association

In [4]:
numeric_predictors = [
    col
    for col in df.select_dtypes(include=np.number).columns
    if col != "fraud_bool"
]

target_correlations = (
    df[numeric_predictors]
    .corrwith(df["fraud_bool"], method="pearson")
)

numeric_leakage_screen = pd.DataFrame({
    "column": target_correlations.index,
    "pearson_correlation": target_correlations.values,
})

numeric_leakage_screen["absolute_correlation"] = (
    numeric_leakage_screen["pearson_correlation"].abs()
)

numeric_leakage_screen["unique_count"] = [
    df[col].nunique(dropna=False)
    for col in numeric_leakage_screen["column"]
]

numeric_leakage_screen = numeric_leakage_screen.sort_values(
    "absolute_correlation",
    ascending=False
)

display(numeric_leakage_screen)

c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\.venv\Lib\site-packages\numpy\lib\_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


,column,pearson_correlation,absolute_correlation,unique_count
13,credit_risk_score,0.070624,0.070624,551
19,proposed_credit_limit,0.068907,0.068907,12
4,customer_age,0.062959,0.062959,9
22,keep_alive_session,-0.050296,0.050296,2
0,income,0.045079,0.045079,9
12,date_of_birth_distinct_emails_4w,-0.043224,0.043224,40
1,name_email_similarity,-0.036720,0.036720,998861
23,device_distinct_emails_8w,0.035704,0.035704,4
18,has_other_cards,-0.035156,0.035156,2
15,phone_home_valid,-0.035128,0.035128,2


## Numerical Leakage Screen Interpretation

The numerical predictors were screened for unusually strong linear relationships with `fraud_bool` using Pearson correlation.

No numerical feature showed an extremely high correlation with the target.

The largest absolute correlations were approximately:

- `credit_risk_score`: 0.071
- `proposed_credit_limit`: 0.069
- `customer_age`: 0.063
- `keep_alive_session`: 0.050
- `income`: 0.045

These values do not indicate an obvious numerical feature that directly reproduces the fraud label.

However, this screen does not prove that the features are leakage-safe. Leakage depends primarily on when and how a feature is generated.

`credit_risk_score` therefore remains a high-priority semantic review item despite its relatively small correlation.

`device_fraud_count` produced an undefined (`NaN`) Pearson correlation because the feature is constant at zero in `Base.csv`. A constant variable has zero variance, so correlation cannot be calculated.

The relatively small correlation between `month` and `fraud_bool` also does not rule out temporal shift. Milestone 01 showed that fraud prevalence changes meaningfully across individual months, demonstrating that aggregate linear correlation can miss ordered temporal behavior.

No feature will be approved or excluded solely because of this correlation screen.

### 4. Screen numerical features for drift across month

In [5]:
temporal_numeric_columns = [
    col
    for col in df.select_dtypes(include=np.number).columns
    if col not in ["fraud_bool", "month", "device_fraud_count"]
]

monthly_medians = (
    df.groupby("month")[temporal_numeric_columns]
    .median()
)

temporal_drift_rows = []

for col in temporal_numeric_columns:
    q1 = df[col].quantile(0.25)
    q3 = df[col].quantile(0.75)
    iqr = q3 - q1

    month_min_median = monthly_medians[col].min()
    month_max_median = monthly_medians[col].max()

    normalized_median_range = (
        (month_max_median - month_min_median) / iqr
        if iqr != 0
        else np.nan
    )

    temporal_drift_rows.append({
        "column": col,
        "month_0_median": monthly_medians.loc[0, col],
        "month_7_median": monthly_medians.loc[7, col],
        "minimum_month_median": month_min_median,
        "maximum_month_median": month_max_median,
        "iqr": iqr,
        "normalized_median_range": normalized_median_range,
        "absolute_normalized_median_range": abs(normalized_median_range),
    })

numeric_temporal_drift = (
    pd.DataFrame(temporal_drift_rows)
    .sort_values(
        "absolute_normalized_median_range",
        ascending=False
    )
)

display(numeric_temporal_drift)

,column,month_0_median,month_7_median,minimum_month_median,maximum_month_median,iqr,normalized_median_range,absolute_normalized_median_range
10,velocity_4w,6343.590545,3122.498634,3122.498634,6343.590545,1219.714934,2.640856,2.640856
9,velocity_24h,5950.369216,3026.777525,3026.777525,5950.369216,2159.395056,1.353894,1.353894
15,phone_home_valid,0.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
14,email_is_free,1.000000,0.000000,0.000000,1.000000,1.000000,1.000000,1.000000
22,keep_alive_session,1.000000,1.000000,0.000000,1.000000,1.000000,1.000000,1.000000
8,velocity_6h,7623.045581,3462.117400,3462.117400,7623.045581,4244.351979,0.980345,0.980345
7,zip_count_4w,1702.000000,844.000000,844.000000,1702.000000,1050.000000,0.817143,0.817143
12,date_of_birth_distinct_emails_4w,12.000000,7.000000,7.000000,12.000000,7.000000,0.714286,0.714286
13,credit_risk_score,106.000000,125.000000,99.000000,139.000000,95.000000,0.421053,0.421053
0,income,0.600000,0.700000,0.500000,0.700000,0.500000,0.400000,0.400000


### Inspect the strongest continuous temporal shifts month by month

In [6]:
top_temporal_numeric_features = [
    "velocity_4w",
    "velocity_24h",
    "velocity_6h",
    "zip_count_4w",
    "date_of_birth_distinct_emails_4w",
    "credit_risk_score",
]

top_feature_monthly_medians = (
    df.groupby("month")[top_temporal_numeric_features]
    .median()
)

display(top_feature_monthly_medians)

,velocity_4w,velocity_24h,velocity_6h,zip_count_4w,date_of_birth_distinct_emails_4w,credit_risk_score
month,,,,,,
0,6343.590545,5950.369216,7623.045581,1702.0,12.0,106.0
1,5446.323889,5340.661312,6482.067044,1333.0,10.0,99.0
2,5462.167155,5044.822646,6164.507328,1412.0,10.0,106.0
3,4909.927613,4969.445524,5940.098864,1354.0,9.0,138.0
4,4695.658606,4662.340178,5242.324448,1301.0,9.0,130.0
5,4267.070964,4215.330451,4620.621080,1142.0,8.0,133.0
6,4217.065599,3354.770176,3772.525990,1075.0,7.0,139.0
7,3122.498634,3026.777525,3462.117400,844.0,7.0,125.0


## Month & Month Temporal Drift Interpretation
This table gives us much stronger evidence of real temporal drift.
Several features move in a clear direction across the eight months rather than just bouncing randomly. The three velocity measures decline substantially from month 0 to month 7, zip_count_4w drops from 1702 to 844, and date_of_birth_distinct_emails_4w falls from 12 to 7. credit_risk_score changes too, but less smoothly.
That pattern makes a chronological holdout increasingly defensible because later months look meaningfully different from earlier months.

### Measure temporal drift for binary indicators

In [7]:
binary_temporal_columns = [
    "email_is_free",
    "phone_home_valid",
    "phone_mobile_valid",
    "has_other_cards",
    "foreign_request",
    "keep_alive_session",
]

monthly_binary_rates = (
    df.groupby("month")[binary_temporal_columns]
    .mean()
    * 100
)

display(monthly_binary_rates)

,email_is_free,phone_home_valid,phone_mobile_valid,has_other_cards,foreign_request,keep_alive_session
month,,,,,,
0,57.154938,41.264724,89.692691,17.075657,3.970855,67.758230
1,58.787024,42.306065,85.995925,20.877605,2.075693,59.064410
2,52.879638,36.044211,87.208258,17.519474,2.228079,55.986684
3,55.232019,34.646473,88.994673,27.232072,2.268511,52.046563
4,49.014418,40.009084,88.049275,24.430069,2.655630,48.051155
5,51.739396,50.655783,91.281647,27.414664,2.420321,59.248427
6,47.046261,44.745211,91.381924,24.020967,2.298277,57.094520
7,49.723780,48.362814,90.001342,19.348843,2.161230,64.817282


## Binary Indicators Temporal Drift
The binary indicators also show temporal movement, although the pattern is less consistently monotonic than the continuous velocity features.
A few shifts are still meaningful. email_is_free falls from about 57.2% in month 0 to around 49.7% in month 7. has_other_cards rises sharply in the middle months before dropping again. keep_alive_session also changes materially over time, from about 67.8% in month 0 to a low near 48.1% in month 4, then back above 64% by month 7. These changes reinforce that the data distribution is evolving across months rather than remaining stationary.

### Measure temporal drift for categorical features

In [8]:
categorical_temporal_columns = [
    "payment_type",
    "employment_status",
    "housing_status",
    "source",
    "device_os",
]

for col in categorical_temporal_columns:
    monthly_distribution = (
        pd.crosstab(
            df["month"],
            df[col],
            normalize="index"
        )
        * 100
    )

    print(f"\n--- Monthly distribution for {col} (%) ---")
    display(monthly_distribution.round(2))


--- Monthly distribution for payment_type (%) ---


payment_type,AA,AB,AC,AD,AE
month,,,,,
0,30.96,33.84,13.78,21.38,0.04
1,25.73,33.98,28.46,11.80,0.03
2,27.33,30.83,31.46,10.34,0.03
3,28.68,36.08,25.54,9.65,0.04
4,20.94,39.57,27.64,11.82,0.03
5,23.15,44.23,22.68,9.93,0.01
6,28.80,38.48,24.25,8.46,0.01
7,18.77,42.08,28.15,10.98,0.03



--- Monthly distribution for employment_status (%) ---


employment_status,CA,CB,CC,CD,CE,CF,CG
month,,,,,,,
0,64.85,22.33,3.37,2.86,2.06,4.49,0.04
1,69.49,16.71,3.66,3.14,3.06,3.91,0.03
2,73.87,13.15,4.01,2.71,1.96,4.27,0.03
3,73.10,12.82,4.83,2.82,1.44,4.93,0.07
4,74.14,10.96,4.46,3.07,2.43,4.87,0.06
5,76.07,11.06,3.04,1.95,2.99,4.85,0.04
6,77.50,10.69,3.93,1.85,2.12,3.87,0.04
7,77.32,11.64,2.34,2.59,2.33,3.74,0.04



--- Monthly distribution for housing_status (%) ---


housing_status,BA,BB,BC,BD,BE,BF,BG
month,,,,,,,
0,16.80,17.98,41.84,3.10,20.03,0.19,0.06
1,13.76,28.21,32.89,2.02,22.86,0.24,0.02
2,10.90,31.31,40.06,1.65,15.90,0.15,0.03
3,19.54,25.07,42.74,1.68,10.81,0.13,0.02
4,20.51,24.36,36.13,2.04,16.77,0.18,0.02
5,16.22,27.61,32.66,3.25,20.11,0.13,0.02
6,19.15,30.66,30.29,3.45,16.27,0.16,0.02
7,19.81,23.97,38.72,4.62,12.73,0.14,0.01



--- Monthly distribution for source (%) ---


source,INTERNET,TELEAPP
month,,
0,99.43,0.57
1,99.16,0.84
2,99.49,0.51
3,99.17,0.83
4,98.99,1.01
5,99.44,0.56
6,99.14,0.86
7,99.61,0.39



--- Monthly distribution for device_os (%) ---


device_os,linux,macintosh,other,windows,x11
month,,,,,
0,32.79,4.02,38.24,24.17,0.78
1,37.75,4.52,29.80,27.40,0.52
2,39.64,4.70,28.06,26.89,0.71
3,32.43,6.01,33.97,26.79,0.80
4,29.29,7.22,32.75,29.90,0.85
5,29.69,6.04,37.89,25.56,0.83
6,29.05,5.44,41.20,23.52,0.79
7,34.70,5.09,33.80,25.96,0.44


### Categorical Feature Temporal Drift Interpretation
The categorical features also show clear temporal movement, which strengthens the case that the eight months are not identically distributed.
A few examples stand out. payment_type changes substantially: AB rises from about 33.8% to 42.1%, while AD falls from about 21.4% to 11.0%. employment_status=CA rises from about 64.9% to 77.3%, while CB roughly halves. housing_status and device_os also move materially across months. source is comparatively stable and remains overwhelmingly INTERNET.
So far we now have temporal movement in fraud prevalence, continuous variables, binary indicators, and categorical composition. That is strong evidence that a random split could make evaluation easier than the real deployment setting by mixing earlier and later distributions together.

### Quantify categorical drift

In [9]:
categorical_drift_rows = []

for col in categorical_temporal_columns:
    monthly_distribution = (
        pd.crosstab(
            df["month"],
            df[col],
            normalize="index"
        )
        * 100
    )

    category_ranges = (
        monthly_distribution.max(axis=0)
        - monthly_distribution.min(axis=0)
    )

    most_shifted_category = category_ranges.idxmax()
    largest_range = category_ranges.max()

    categorical_drift_rows.append({
        "column": col,
        "most_shifted_category": most_shifted_category,
        "largest_percentage_point_range": largest_range,
    })

categorical_temporal_drift = (
    pd.DataFrame(categorical_drift_rows)
    .sort_values(
        "largest_percentage_point_range",
        ascending=False
    )
)

display(categorical_temporal_drift)

,column,most_shifted_category,largest_percentage_point_range
0,payment_type,AC,17.683367
2,housing_status,BB,13.329716
4,device_os,other,13.144794
1,employment_status,CA,12.647298
3,source,INTERNET,0.617579


## Temporal Drift Interpretation

The dataset shows meaningful temporal distribution shift across the eight values of `month`.

### Continuous and Numerical Features

Several numerical features change substantially across time.

The strongest shifts were observed in:

- `velocity_4w`
- `velocity_24h`
- `velocity_6h`
- `zip_count_4w`
- `date_of_birth_distinct_emails_4w`

The velocity features generally decline from early months to later months, indicating a clear change in the underlying data distribution.

### Binary Features

Several binary indicators also change in prevalence across months, including:

- `email_is_free`
- `phone_home_valid`
- `has_other_cards`
- `keep_alive_session`

These changes are not always monotonic, but they show that the proportion of observations in each state is not constant over time.

### Categorical Features

The categorical variables also show substantial temporal changes.

The largest observed category-prevalence ranges were approximately:

- `payment_type`: 17.7 percentage points
- `housing_status`: 13.3 percentage points
- `device_os`: 13.1 percentage points
- `employment_status`: 12.6 percentage points

`source` was comparatively stable.

### Split Implication

Because fraud prevalence and multiple predictor distributions change across time, a random split would mix earlier and later observations across training, validation, and test sets.

That could produce an evaluation that is more optimistic than a real deployment scenario.

The evidence therefore supports using a chronological split as the primary split strategy, subject to final verification of month ordering and adequate fraud representation in each partition.

### Evaluate candidate chronological partitions

In [10]:
candidate_split = pd.Series(index=df.index, dtype="object")

candidate_split[df["month"].isin([0, 1, 2, 3, 4, 5])] = "train"
candidate_split[df["month"] == 6] = "validation"
candidate_split[df["month"] == 7] = "test"

candidate_split_summary = (
    pd.DataFrame({
        "split": candidate_split,
        "fraud_bool": df["fraud_bool"],
    })
    .groupby("split")["fraud_bool"]
    .agg(
        total_records="count",
        fraud_cases="sum",
        fraud_rate="mean",
    )
)

candidate_split_summary["record_percent"] = (
    candidate_split_summary["total_records"] / len(df) * 100
)

candidate_split_summary["fraud_percent"] = (
    candidate_split_summary["fraud_rate"] * 100
)

candidate_split_summary = candidate_split_summary.loc[
    ["train", "validation", "test"]
]

display(candidate_split_summary)

,total_records,fraud_cases,fraud_rate,record_percent,fraud_percent
split,,,,,
train,794989,8151,0.010253,79.4989,1.025297
validation,108168,1450,0.013405,10.8168,1.340507
test,96843,1428,0.014746,9.6843,1.474552


### Validate the chronological split boundaries

In [11]:
split_validation = (
    pd.DataFrame({
        "split": candidate_split,
        "month": df["month"],
        "fraud_bool": df["fraud_bool"],
    })
    .groupby("split")
    .agg(
        min_month=("month", "min"),
        max_month=("month", "max"),
        total_records=("month", "size"),
        fraud_cases=("fraud_bool", "sum"),
    )
    .loc[["train", "validation", "test"]]
)

display(split_validation)

train_max_month = split_validation.loc["train", "max_month"]
validation_min_month = split_validation.loc["validation", "min_month"]
validation_max_month = split_validation.loc["validation", "max_month"]
test_min_month = split_validation.loc["test", "min_month"]

assert train_max_month < validation_min_month
assert validation_max_month < test_min_month

print("Chronological ordering check: PASS")
print("No month overlap between train, validation, and test.")

,min_month,max_month,total_records,fraud_cases
split,,,,
train,0,5,794989,8151
validation,6,6,108168,1450
test,7,7,96843,1428


Chronological ordering check: PASS
No month overlap between train, validation, and test.


## Final Split Decision

The project will use a chronological train/validation/test split.

### Partition Definition

| Partition | Months | Records | Fraud Cases | Fraud Rate |
|---|---|---:|---:|---:|
| Train | 0–5 | 794,989 | 8,151 | 1.0253% |
| Validation | 6 | 108,168 | 1,450 | 1.3405% |
| Test | 7 | 96,843 | 1,428 | 1.4746% |

This corresponds to approximately:

- 79.5% training
- 10.8% validation
- 9.7% final test

### Rationale

A chronological split was selected because both the target prevalence and multiple predictor distributions change across `month`.

A random stratified split would mix earlier and later observations across all partitions and could hide temporal distribution shift.

The chronological design better represents the intended deployment scenario:

1. Train on historical applications.
2. Make development decisions using a later validation period.
3. Evaluate once on the newest unseen period.

### Final Test Set Policy

Month 7 is now designated as the sealed final test period.

The test partition must not be used for:

- feature selection
- preprocessing decisions
- imputation decisions
- model selection
- hyperparameter tuning
- class-imbalance strategy selection
- probability calibration
- threshold tuning
- business-cost optimization

The test partition will be evaluated only after the preprocessing pipeline, model architecture, hyperparameters, calibration method, threshold policy, and evaluation procedure have been frozen.

### Validation Policy

Month 6 may be used for development decisions, including:

- model comparison
- hyperparameter selection
- early stopping
- calibration selection
- threshold selection
- error analysis

### Training Policy

Any learned preprocessing statistics must be fit using months 0–5 only.

Validation and test data may only be transformed using preprocessing learned from the training partition.

### Time Feature Policy

`month` will be used for split assignment, temporal analysis, and monitoring.

It will not be used as a predictive model feature in the initial modeling pipeline. This avoids allowing the model to rely directly on the dataset's absolute time index.

### Reproducibility Rule

Split membership is deterministic:

- `month <= 5` → train
- `month == 6` → validation
- `month == 7` → test

No random seed is required to reproduce the primary split.

### Create a machine-readable split manifest

In [12]:
import json

SPLIT_CONFIG_PATH = (
    PROJECT_ROOT
    / "configs"
    / "split_config.json"
)

split_config = {
    "version": "1.0",
    "dataset": "Base.csv",
    "target": "fraud_bool",
    "temporal_field": "month",
    "strategy": "chronological",
    "train": {
        "months": [0, 1, 2, 3, 4, 5],
        "rule": "month <= 5",
        "records": int((df["month"] <= 5).sum()),
        "fraud_cases": int(
            df.loc[df["month"] <= 5, "fraud_bool"].sum()
        ),
    },
    "validation": {
        "months": [6],
        "rule": "month == 6",
        "records": int((df["month"] == 6).sum()),
        "fraud_cases": int(
            df.loc[df["month"] == 6, "fraud_bool"].sum()
        ),
    },
    "test": {
        "months": [7],
        "rule": "month == 7",
        "records": int((df["month"] == 7).sum()),
        "fraud_cases": int(
            df.loc[df["month"] == 7, "fraud_bool"].sum()
        ),
        "sealed": True,
    },
    "month_used_as_model_feature_initially": False,
}

with open(SPLIT_CONFIG_PATH, "w", encoding="utf-8") as f:
    json.dump(
        split_config,
        f,
        indent=2
    )

print("Saved split configuration to:")
print(SPLIT_CONFIG_PATH)

print("\nSplit configuration:")
print(json.dumps(split_config, indent=2))

print("\nFile exists:", SPLIT_CONFIG_PATH.exists())

Saved split configuration to:
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\configs\split_config.json

Split configuration:
{
  "version": "1.0",
  "dataset": "Base.csv",
  "target": "fraud_bool",
  "temporal_field": "month",
  "strategy": "chronological",
  "train": {
    "months": [
      0,
      1,
      2,
      3,
      4,
      5
    ],
    "rule": "month <= 5",
    "records": 794989,
    "fraud_cases": 8151
  },
  "validation": {
    "months": [
      6
    ],
    "rule": "month == 6",
    "records": 108168,
    "fraud_cases": 1450
  },
  "test": {
    "months": [
      7
    ],
    "rule": "month == 7",
    "records": 96843,
    "fraud_cases": 1428,
    "sealed": true
  },
  "month_used_as_model_feature_initially": false
}

File exists: True


### Export the Milestone 02 leakage review and split metrics

In [13]:
LEAKAGE_REVIEW_PATH = (
    PROJECT_ROOT
    / "reports"
    / "milestone_02_leakage_review.csv"
)

SPLIT_METRICS_PATH = (
    PROJECT_ROOT
    / "reports"
    / "milestone_02_split_metrics.csv"
)

# Export the feature-level leakage review.
leakage_review.to_csv(
    LEAKAGE_REVIEW_PATH,
    index=False,
)

# Build a compact split summary.
split_metrics = candidate_split_summary.reset_index()

split_metrics["min_month"] = split_metrics["split"].map({
    "train": 0,
    "validation": 6,
    "test": 7,
})

split_metrics["max_month"] = split_metrics["split"].map({
    "train": 5,
    "validation": 6,
    "test": 7,
})

split_metrics["sealed"] = split_metrics["split"].map({
    "train": False,
    "validation": False,
    "test": True,
})

split_metrics.to_csv(
    SPLIT_METRICS_PATH,
    index=False,
)

print("Saved leakage review:")
print(LEAKAGE_REVIEW_PATH)

print("\nSaved split metrics:")
print(SPLIT_METRICS_PATH)

print("\nLeakage review rows:", len(leakage_review))
print("Split metric rows:", len(split_metrics))

print("\nFiles exist:")
print("Leakage review:", LEAKAGE_REVIEW_PATH.exists())
print("Split metrics:", SPLIT_METRICS_PATH.exists())

display(split_metrics)

Saved leakage review:
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_02_leakage_review.csv

Saved split metrics:
c:\Users\vrber\ML_Projects\bank-account-fraud-dl-mlops\reports\milestone_02_split_metrics.csv

Leakage review rows: 32
Split metric rows: 3

Files exist:
Leakage review: True
Split metrics: True


,split,total_records,fraud_cases,fraud_rate,record_percent,fraud_percent,min_month,max_month,sealed
0,train,794989,8151,0.010253,79.4989,1.025297,0,5,False
1,validation,108168,1450,0.013405,10.8168,1.340507,6,6,False
2,test,96843,1428,0.014746,9.6843,1.474552,7,7,True


## Milestone 02 Conclusion

Milestone 02 evaluated leakage risk, temporal distribution shift, and train/validation/test split design for the Bank Account Fraud dataset.

### Leakage Review

No numerical predictor showed an unusually strong linear correlation with `fraud_bool`, so the empirical screen did not reveal an obvious direct numerical proxy for the target.

However, correlation alone cannot establish whether a feature is leakage-safe. Prediction-time availability and feature-generation semantics remain the primary considerations.

Key decisions and review items include:

- `fraud_bool` is the target and is always excluded from model inputs.
- `device_fraud_count` is constant at zero in `Base.csv` and is a drop candidate.
- `device_fraud_count` also deserves semantic caution because its name references prior fraudulent applications.
- `credit_risk_score` remains a high-priority feature-definition review item because it is a derived risk score.
- `month` is reserved for temporal splitting, analysis, and monitoring and will not initially be used as a predictive model feature.
- Other candidate predictors remain subject to prediction-time availability requirements.

### Temporal Drift

Meaningful temporal drift was observed across multiple feature types.

Continuous features with strong temporal movement included:

- `velocity_4w`
- `velocity_24h`
- `velocity_6h`
- `zip_count_4w`
- `date_of_birth_distinct_emails_4w`

Binary indicators also changed in prevalence across months.

Categorical variables showed substantial changes in category composition, particularly:

- `payment_type`
- `housing_status`
- `device_os`
- `employment_status`

Fraud prevalence also increased in the later periods.

These findings indicate that randomly mixing observations across months could hide real temporal distribution shift.

### Final Split Design

The primary project split is therefore chronological:

- Months 0–5 → Training
- Month 6 → Validation
- Month 7 → Final Test

This produces:

- 794,989 training records with 8,151 fraud cases
- 108,168 validation records with 1,450 fraud cases
- 96,843 test records with 1,428 fraud cases

The partitions have no temporal overlap.

### Final Test Policy

Month 7 is now the sealed final test period.

It must not be used for:

- feature selection
- preprocessing decisions
- hyperparameter tuning
- model selection
- class-imbalance strategy selection
- calibration
- threshold selection
- business-cost optimization

The test set will be evaluated only after the complete development procedure has been frozen.

### Reproducibility

The split rules are stored in:

`configs/split_config.json`

Reusable split logic is implemented in:

`src/data/splitting.py`

Automated tests verify deterministic split assignment and guard against missing or unexpected month values.

### Next Milestone

Milestone 03 will build the leakage-safe preprocessing and PyTorch data pipeline.

All learned preprocessing operations must be fit on the training partition only and then applied unchanged to validation and test data.